# real-chart-bench: LineFormer pretrained baseline (Google Colab)

Runs the LineFormer pretrained model (ICDAR2023, arxiv 2305.01837) against the
**real-chart-bench** v0 verified-image evaluation suite and writes
`results/lineformer-pretrained.json` in the project's standard schema.

**Why Colab, not local**: `mmcv` ships source-only on PyPI and OpenMMLab's
prebuilt wheels are Linux+CUDA only. That is exactly what a Colab GPU runtime
provides for free, so this notebook targets Colab instead of fighting a local
macOS/Python-3.14 environment (see `docs/design/benchmark-architecture.md`
§7.16 for the local-infeasibility writeup, §7.19 for this notebook's context).

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (or better). CPU also works but is slow.
2. The repo (`t29mato/real-chart-bench`) is **public** — no GitHub token needed.
3. Run cells **top to bottom, in order, in a single session**. Cell 2 installs
   the package with a regular (non-editable) `pip install .` — editable
   installs (`-e .`) do NOT reliably import in the same kernel session
   without a restart (confirmed failure mode, fixed 2026-08-19, see design
   §7.26) — and prints an explicit `OK: real_chart_bench installed at ...`
   confirmation. If that line doesn't print, don't run anything below it;
   re-run Cell 2 (Runtime → Restart session first, if it still fails).
4. Estimated free-tier runtime: 5–15 minutes (dominated by mmcv/mmdet install,
   not inference — the eval set is tiny, see the verification-gate note below).

**Scope note**: real-image evaluation is gated on `data/verified_pairs/registry.json`
(§7.19: "量より信頼性。ベンチマークの信用が資産" — reliability over quantity). Only
`status: "verified"` entries are used; this is intentionally a small, trustworthy
set rather than the full (unverified) image↔ground-truth pairing.

## 1. Clone real-chart-bench and install it + eval-only deps

In [ ]:
# real-chart-bench is now PUBLIC — no token needed.
!rm -rf /content/real-chart-bench
!git clone --depth 1 https://github.com/t29mato/real-chart-bench.git /content/real-chart-bench
%cd /content/real-chart-bench

# Regular install, NOT editable (-e). Editable installs (PEP 660) work by
# writing a .pth file that maps back to src/ -- but Python's `site` module
# only processes .pth files at *interpreter startup*. A Colab/Jupyter kernel
# is one long-lived interpreter, so `%pip install -e .` succeeds and then
# `import real_chart_bench` in the SAME session still raises
# ModuleNotFoundError (confirmed twice by the owner; reproduced locally in a
# clean venv while fixing this cell -- see design §7.19/§7.26). A regular
# install copies the package straight into site-packages, which is on
# sys.path immediately, no restart needed. This repo doesn't need live-edit
# semantics here (it's a fresh git clone per run), so there's no downside.
#
# %pip (not !pip) installs into the *running kernel's* environment. In some
# Colab configurations !pip can target a different interpreter than the
# kernel, which is a separate, unrelated failure mode this also guards
# against.
%pip install -q .
%pip install -q pymupdf requests

# Fail loudly, right here, if the install didn't actually take -- instead of
# a confusing ModuleNotFoundError several cells downstream. If this cell
# doesn't print the OK line, do not run anything below it.
import importlib

importlib.invalidate_caches()
import real_chart_bench  # noqa: F401

print(f"OK: real_chart_bench installed at {real_chart_bench.__file__}")


## 2. Install the LineFormer / mmdetection stack

This is exactly the install that fails on macOS (no prebuilt `mmcv` wheel) but
works on Colab's Linux+CUDA runtime via `openmim`.

**⚠️ Assumption flagged**: the LineFormer inference API below (`Cell 5`) is
written from the paper's public repo README as of when this notebook was
authored, but was **not executed/verified end-to-end** in this session (no
Colab access from the coding environment that wrote this notebook). If
`TheJaeLal/LineFormer`'s API has since changed, check its README against
Cell 5 and adjust the `LineFormerModelRunner.extract()` body — the rest of
the notebook (data prep, scoring, results schema) does not depend on that
detail and should not need changes.

In [ ]:
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q "mmcv>=2.0.0"
!mim install -q mmdet

!git clone --depth 1 https://github.com/TheJaeLal/LineFormer.git /content/LineFormer
%cd /content/LineFormer
!pip install -q -r requirements.txt || true

# Pretrained checkpoint: see the LineFormer README's "Pretrained Models" section
# for the current download link (Google Drive / OneDrive, not a stable pip
# artifact) and update CHECKPOINT_URL below if the link has moved.
CHECKPOINT_URL = ""  # fill in from the LineFormer README before running
CHECKPOINT_PATH = "/content/LineFormer/checkpoints/lineformer.pth"
CONFIG_PATH = "/content/LineFormer/lineformer_swin_t_config.py"  # adjust to the repo's actual config filename

import os
os.makedirs("/content/LineFormer/checkpoints", exist_ok=True)
if CHECKPOINT_URL:
    !wget -q -O {CHECKPOINT_PATH} "{CHECKPOINT_URL}"
else:
    print("CHECKPOINT_URL is empty — set it from the LineFormer README before running inference.")

## 3. Reconstruct the verified-pairs images

`data/raw/images/` is gitignored (546MB, regeneratable), so this notebook
re-fetches just the papers referenced by `data/verified_pairs/registry.json`
(currently 1 VERIFIED entry) using the same adapters the collection pipeline
uses — not a bulk 603-paper re-run. Single-paper, single PDF fetch: well
within politeness norms for the source publisher server.

In [ ]:
%cd /content/real-chart-bench
import json
import pathlib

from real_chart_bench.adapter.verified_pairing_registry import load_registry
from real_chart_bench.usecase.real_image_gate import select_verified_pairings

REPO_ROOT = pathlib.Path("/content/real-chart-bench")
registry = load_registry(REPO_ROOT / "data/verified_pairs/registry.json")
verified = select_verified_pairings(registry)
print(f"{len(verified)} VERIFIED pairing(s) will be evaluated:")
for p in verified:
    print(f"  paper {p.paper_id}, figure {p.figure_id}, panel {p.panel_label!r}")

In [ ]:
# papers.json does not store pdf_url (it's small committed metadata, design
# data-layout convention) -- re-resolve it from OpenAlex via the paper's DOI,
# exactly like scripts/collect/collect_v0_dataset.py's classify_all_papers()
# does, then re-run the real PdfFetchPort/FigureExtractionPort adapters for
# just these papers (not a bulk re-collection).
import urllib.parse
import urllib.request

from real_chart_bench.adapter.pdf_fetch import HttpPdfFetchAdapter
from real_chart_bench.adapter.figure_extraction import PyMuPdfFigureExtractor
from real_chart_bench.adapter.panel_layout import PyMuPdfPanelSplitter
from real_chart_bench.usecase.pdf_fetch import PdfFetchStatus

papers = json.loads((REPO_ROOT / "data/manifest/v0/papers.json").read_text())
papers_by_id = {p["paper_id"]: p for p in papers}


def resolve_pdf_url(doi: str) -> str | None:
    params = urllib.parse.urlencode({"filter": f"doi:{doi}", "per-page": 1})
    req = urllib.request.Request(
        f"https://api.openalex.org/works?{params}",
        headers={"User-Agent": "real-chart-bench/0.0.1 (mailto:tomoya.matou@gmail.com)"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.load(resp)
    results = data.get("results") or []
    if not results:
        return None
    work = results[0]
    best_oa = (work.get("best_oa_location") or {})
    primary = (work.get("primary_location") or {})
    return best_oa.get("pdf_url") or primary.get("pdf_url")


pdf_fetcher = HttpPdfFetchAdapter()
extractor = PyMuPdfFigureExtractor()

IMAGES_DIR = pathlib.Path("/content/images")
IMAGES_DIR.mkdir(exist_ok=True)

images_by_paper: dict[str, bytes] = {}
for pairing in verified:
    paper = papers_by_id[pairing.paper_id]
    pdf_url = resolve_pdf_url(paper["doi"])
    if not pdf_url:
        raise RuntimeError(f"no pdf_url resolvable for paper {pairing.paper_id} (doi={paper['doi']})")

    fetch_result = pdf_fetcher.fetch(pdf_url)
    if fetch_result.status is not PdfFetchStatus.OK or not fetch_result.content:
        raise RuntimeError(f"PDF fetch failed for paper {pairing.paper_id}: {fetch_result.status}")

    extracted = extractor.extract(fetch_result.content)
    # Same naming convention as collect_v0_dataset.py, so pairing.image_path
    # (e.g. 'p04_embedded_4.jpg') addresses the same extracted image again.
    named = {}
    for i, img in enumerate(extracted):
        ext = "png" if img.source.value == "page_render" else "jpg"
        named[f"p{img.page_number:02d}_{img.source.value}_{i}.{ext}"] = img.image_bytes
    if pairing.image_path not in named:
        raise RuntimeError(
            f"expected image {pairing.image_path!r} not found among {len(named)} re-extracted "
            f"images for paper {pairing.paper_id}; PDF may have changed or extractor settings drifted"
        )
    images_by_paper[pairing.paper_id] = named[pairing.image_path]
    (IMAGES_DIR / f"{pairing.paper_id}_{pairing.image_path}").write_bytes(named[pairing.image_path])

print(f"fetched {len(images_by_paper)} image(s) into {IMAGES_DIR}")

## 4. Build the DatasetItems (real verified pairs + synthetic fixtures)

Mirrors `scripts/eval/run_baselines.py` exactly, so LineFormer's results are
directly comparable to the naive-CV baseline on the same figures.

In [ ]:
import csv, gzip

from real_chart_bench.domain.curve import Curve, ScaleType
from real_chart_bench.usecase.evaluate_dataset import DatasetItem
from real_chart_bench.usecase.model_runner import ExtractionTask


def ground_truth_for(figure_id: str) -> list[Curve]:
    rows = []
    with gzip.open(REPO_ROOT / "data/cache/ThermoelectricMaterials_curves.csv.gz", "rt") as f:
        for row in csv.DictReader(f):
            if row["figure_id"] == figure_id:
                rows.append(row)
    return [
        Curve(x_values=tuple(json.loads(r["x"])), y_values=tuple(json.loads(r["y"])), series_label=r["prop_y"])
        for r in rows
    ]


real_items = []
splitter = PyMuPdfPanelSplitter()
for pairing in verified:
    image_bytes = images_by_paper[pairing.paper_id]
    if pairing.panel_label is not None:
        panels = {p.label: p for p in splitter.split(image_bytes)}
        image_bytes = panels[pairing.panel_label].image_bytes
    task = ExtractionTask(
        image_bytes=image_bytes,
        x_range=pairing.x_range,
        y_range=pairing.y_range,
        x_scale=pairing.x_scale,
        y_scale=pairing.y_scale,
    )
    real_items.append(
        DatasetItem(
            figure_id=f"{pairing.paper_id}-{pairing.figure_id}",
            task=task,
            ground_truth=ground_truth_for(pairing.figure_id),
        )
    )

print(f"{len(real_items)} real DatasetItem(s) built")

In [ ]:
import pymupdf


def synthetic_items() -> list[DatasetItem]:
    items = []

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(1, 0, 0), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-linear-single",
        task=ExtractionTask(image_bytes=png, x_range=(0, 10), y_range=(0, 10)),
        ground_truth=[Curve(x_values=(0.0, 10.0), y_values=(10.0, 0.0))],
    ))

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 280), pymupdf.Point(280, 20), color=(1, 0, 0), width=2)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(0, 0, 1), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-linear-two-series",
        task=ExtractionTask(image_bytes=png, x_range=(0, 10), y_range=(0, 10)),
        ground_truth=[
            Curve(x_values=(0.0, 10.0), y_values=(0.0, 10.0), series_label="up"),
            Curve(x_values=(0.0, 10.0), y_values=(10.0, 0.0), series_label="down"),
        ],
    ))

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(0, 0, 0), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-log-black-line",
        task=ExtractionTask(image_bytes=png, x_range=(1, 100), y_range=(0, 10), x_scale=ScaleType.LOG),
        ground_truth=[Curve(x_values=(1.0, 100.0), y_values=(10.0, 0.0), x_scale=ScaleType.LOG)],
    ))

    return items


dataset_items = real_items + synthetic_items()
print(f"{len(dataset_items)} total DatasetItem(s)")

## 5. LineFormer ModelRunner adapter

Implements the same `ModelRunnerPort` protocol as `NaiveCvModelRunner` /
`LlmModelRunner` (`extract(task: ExtractionTask) -> list[Curve]`), so it plugs
into the existing `evaluate_model_on_dataset()` usecase unchanged. Pixel→data
calibration reuses `task.x_range`/`task.y_range` exactly like the naive CV
baseline (v0 scope: axis calibration is *given*, not solved by the model —
see design §3.1).

In [ ]:
import sys
sys.path.insert(0, "/content/LineFormer")

import numpy as np
import cv2

# ADAPT THIS import + call if LineFormer's actual public API differs —
# see /content/LineFormer/README.md for the current entrypoint.
import infer as lineformer_infer  # noqa: E402

from real_chart_bench.domain.curve import Curve
from real_chart_bench.usecase.model_runner import ExtractionTask


class LineFormerModelRunner:
    """Wraps LineFormer's pretrained instance-segmentation model behind the
    project's ModelRunnerPort protocol. LineFormer returns pixel-space line
    traces per detected series; this maps each pixel trace to data space
    using the task's given axis calibration (linear or log, independently
    per axis — design §7.25)."""

    def __init__(self, config_path: str, checkpoint_path: str):
        lineformer_infer.load_model(config_path, checkpoint_path)  # ADAPT to actual API
        self._config_path = config_path
        self._checkpoint_path = checkpoint_path

    @staticmethod
    def _scale_frac(frac: float, lo: float, hi: float, is_log: bool) -> float:
        # Mirrors domain/pixel_calibration.py's PixelCalibration._scale_frac
        # (design §7.25) -- kept as a small local copy here since this cell
        # runs standalone against a bare pip-installed real_chart_bench, not
        # a dev checkout with test-only helpers exposed.
        if is_log:
            return lo * (hi / lo) ** frac
        return lo + frac * (hi - lo)

    def extract(self, task: ExtractionTask) -> list[Curve]:
        arr = np.frombuffer(task.image_bytes, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        height, width = img.shape[:2]

        # ADAPT: replace with LineFormer's actual inference call. Expected
        # shape per the paper: a list of series, each a list of (px, py)
        # points along the detected line, in image pixel coordinates.
        series_list = lineformer_infer.get_dataseries(img, to_clean=True)

        x0, x1 = task.x_range
        y0, y1 = task.y_range
        x_is_log = task.x_scale.name == "LOG"
        y_is_log = task.y_scale.name == "LOG"
        curves = []
        for i, points in enumerate(series_list):
            xs, ys = [], []
            for px, py in points:
                frac_x = px / width
                frac_y = 1.0 - (py / height)  # image y grows downward
                xs.append(self._scale_frac(frac_x, x0, x1, x_is_log))
                ys.append(self._scale_frac(frac_y, y0, y1, y_is_log))
            order = np.argsort(xs)
            xs = [xs[j] for j in order]
            ys = [ys[j] for j in order]
            curves.append(Curve(x_values=tuple(xs), y_values=tuple(ys), series_label=f"series_{i}", x_scale=task.x_scale))
        return curves

## 6. Run evaluation and write results/lineformer-pretrained.json

In [ ]:
from datetime import UTC, datetime

from real_chart_bench.domain.matching import HungarianCurveMatcher
from real_chart_bench.domain.metrics import NormalizedYDistanceMetric
from real_chart_bench.usecase.evaluate_dataset import evaluate_model_on_dataset

model = LineFormerModelRunner(CONFIG_PATH, CHECKPOINT_PATH)
matcher = HungarianCurveMatcher(metric=NormalizedYDistanceMetric())
results = evaluate_model_on_dataset(model, dataset_items, matcher=matcher)

per_figure = [
    {
        "figure_id": r.figure_id,
        "summary_score": r.evaluation.summary_score,
        "match_rate": r.evaluation.match_rate,
        "mean_curve_distance": r.evaluation.mean_curve_distance,
        "mean_coverage_ratio": r.evaluation.mean_coverage_ratio,
        "error": r.error,
    }
    for r in results
]
mean_score = sum(p["summary_score"] for p in per_figure) / len(per_figure)

payload = {
    "model_id": "lineformer-pretrained",
    "model_name": "LineFormer (pretrained, ICDAR2023)",
    "dataset_version": "v0-eval-pilot-2026-08-16",
    "run_at": datetime.now(UTC).isoformat(),
    "n_figures": len(per_figure),
    "mean_summary_score": mean_score,
    "per_figure": per_figure,
}

out_path = pathlib.Path("/content/lineformer-pretrained.json")
out_path.write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))

## 7. Download the result and add it to the repo

This notebook does **not** push to git automatically (same structural-non-
execution pattern as the HF Hub upload guard and the LLM adapter — pushing
results is a deliberate, reviewed action, not a side effect of running a
notebook). Download the file, then on your own machine:

```bash
mv ~/Downloads/lineformer-pretrained.json results/lineformer-pretrained.json
rm results/lineformer-pending.json
python scripts/leaderboard/generate.py
git add results/ site/
git commit -m "results: LineFormer pretrained baseline (Colab run)"
git push origin main
```

In [ ]:
from google.colab import files
files.download(str(out_path))